In [0]:
%pip install kaggle --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os

# Kaggle 인증 정보 설정
os.environ['KAGGLE_USERNAME'] = 'KIMLEEHWA' 
os.environ['KAGGLE_KEY'] = 'KGAT_123d6446643c6abd6f77f6062078555f'

# training 카탈로그 및 klh 스키마 설정
CATALOG = "training"        
SCHEMA = "klh"      
VOLUME_NAME = "my_volume"

# 카탈로그/스키마 선택 및 Volume 생성
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {VOLUME_NAME}")

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}"

# Kaggle에서 데이터 다운로드
os.system(f"kaggle datasets download -d khushikyad001/world-happiness-report -p {VOLUME_PATH} --unzip")

# 저장 확인
display(dbutils.fs.ls(VOLUME_PATH))

Dataset URL: https://www.kaggle.com/datasets/khushikyad001/world-happiness-report
License(s): MIT


100%|██████████| 219k/219k [00:00<00:00, 2.35MB/s]


path,name,size,modificationTime
dbfs:/Volumes/training/klh/my_volume/world_happiness_report.csv,world_happiness_report.csv,552176,1788156987000


In [0]:
# Bronze -> Silver -> Gold Medallion Pipeline
from pyspark.sql.functions import current_timestamp, col, avg, count, round

# 변수 정의 (training.klh 지정)
CATALOG = "training"
SCHEMA = "klh"
VOLUME_NAME = "my_volume"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}"

# 카탈로그 및 스키마 지정
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# ----------------------------------------------------
# 1. Bronze Layer: 원본 그대로 Delta 테이블 저장
# ----------------------------------------------------
df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{VOLUME_PATH}/world_happiness_report.csv")

df_bronze = df_raw \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path"))

df_bronze.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze_world_happiness")

print("1. Bronze Layer 생성 완료!")

# ----------------------------------------------------
# 2. Silver Layer: 데이터 정제 (결측치 및 중복 제거)
# ----------------------------------------------------
df_bronze_read = spark.read.table("bronze_world_happiness")

df_silver = df_bronze_read \
    .filter(
        col("Country").isNotNull() & 
        col("Year").isNotNull() & 
        col("Happiness_Score").isNotNull()
    ) \
    .dropDuplicates(["Country", "Year"]) \
    .filter(col("Happiness_Score") > 0)

df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_world_happiness")

print("2. Silver Layer 생성 완료!")

# ----------------------------------------------------
# 3. Gold Layer: 국가별 평균 행복지수 집계
# ----------------------------------------------------
df_silver_read = spark.read.table("silver_world_happiness")

df_gold = df_silver_read.groupBy("Country") \
    .agg(
        count("Year").alias("total_years"),
        round(avg("Happiness_Score"), 2).alias("avg_happiness_score"),
        round(avg("GDP_per_Capita"), 2).alias("avg_gdp_per_capita")
    )

df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_country_happiness_summary")

print("3. Gold Layer 생성 완료! (노트북 정상 실행완료)")

1. Bronze Layer 생성 완료!
2. Silver Layer 생성 완료!
3. Gold Layer 생성 완료! (노트북 정상 실행완료)


In [0]:
spark.read.table("training.klh.bronze_world_happiness").show(10)

+------------+----+---------------+--------------+--------------+-----------------------+-------+----------+---------------------+-----------------+---------------+----------+-----------------+-----------------+------------+-------------------+-----------------+-------------------------+-------------+-----------------+---------------+----------+-------------------+---------------+--------------------+--------------------+
|     Country|Year|Happiness_Score|GDP_per_Capita|Social_Support|Healthy_Life_Expectancy|Freedom|Generosity|Corruption_Perception|Unemployment_Rate|Education_Index|Population|Urbanization_Rate|Life_Satisfaction|Public_Trust|Mental_Health_Index|Income_Inequality|Public_Health_Expenditure|Climate_Index|Work_Life_Balance|Internet_Access|Crime_Rate|Political_Stability|Employment_Rate|        _ingested_at|        _source_file|
+------------+----+---------------+--------------+--------------+-----------------------+-------+----------+---------------------+-----------------+

In [0]:
spark.read.table("training.klh.silver_world_happiness").show(10)

+---------+----+---------------+--------------+--------------+-----------------------+-------+----------+---------------------+-----------------+---------------+----------+-----------------+-----------------+------------+-------------------+-----------------+-------------------------+-------------+-----------------+---------------+----------+-------------------+---------------+--------------------+--------------------+
|  Country|Year|Happiness_Score|GDP_per_Capita|Social_Support|Healthy_Life_Expectancy|Freedom|Generosity|Corruption_Perception|Unemployment_Rate|Education_Index|Population|Urbanization_Rate|Life_Satisfaction|Public_Trust|Mental_Health_Index|Income_Inequality|Public_Health_Expenditure|Climate_Index|Work_Life_Balance|Internet_Access|Crime_Rate|Political_Stability|Employment_Rate|        _ingested_at|        _source_file|
+---------+----+---------------+--------------+--------------+-----------------------+-------+----------+---------------------+-----------------+---------

In [0]:
spark.read.table("training.klh.gold_country_happiness_summary").show(10)

+------------+-----------+-------------------+------------------+
|     Country|total_years|avg_happiness_score|avg_gdp_per_capita|
+------------+-----------+-------------------+------------------+
|   Australia|         20|               5.85|          34019.77|
|         USA|         20|               6.24|          27032.45|
|      Canada|         20|               5.11|          29526.26|
|South Africa|         20|                5.2|           25718.2|
|       China|         20|               5.45|          29286.92|
|       India|         20|               5.66|           31066.6|
|          UK|         20|               5.39|          37711.15|
|      Brazil|         20|               5.57|          30321.88|
|      France|         20|               5.63|          38581.95|
|     Germany|         20|               5.61|          28695.81|
+------------+-----------+-------------------+------------------+



In [0]:
from pyspark.sql.functions import col

df_bronze = spark.read.table("training.klh.bronze_world_happiness")

duplicate_counts = df_bronze.groupBy("Country", "Year").count()

duplicates_only = duplicate_counts.filter(col("count") > 1).orderBy(col("count").desc())

display(duplicates_only)

Country,Year,count
India,2017,32
Australia,2016,32
Brazil,2021,32
Canada,2022,31
Brazil,2019,30
South Africa,2012,28
France,2020,28
Germany,2014,28
France,2013,27
UK,2019,27
